# 01 — Filter significant splicing events

This notebook filters SUPPA's differential splicing results to find
significant events, and then identifies which events are unique to
variable boundary mode compared to strict.

**Filter criteria:** p-value < 0.05 AND |dPSI| ≥ 0.1

**Events analysed:** A3, A5, RI

**Comparisons:** CT8 vs CT20 and CT20 vs PerKO

**Runs compared:** Strict vs Variable 1nt and Variable 5nt

In [53]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# Diff output paths
STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/diff"
VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/diff"
VAR5   = "/Users/gricey/Desktop/Internship/data/output_variable_5nt/diff"

# IOE paths
IOE_STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/events"
IOE_VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/events"
IOE_VAR5   = "/Users/gricey/Desktop/Internship/data/output_variable_5nt/events"

# Events and comparison
EVENTS = ["A3", "A5", "RI"]
COMPARISONS = {"CT8_vs_CT20": 0, "CT20_vs_PerKO": 1}

# --- Helper functions ---

def load_and_filter(diff_dir, event, temp=0):
    path = f"{diff_dir}/diff_{event}.dpsi.temp.{temp}"
    df = pd.read_csv(path, sep="\t")
    df.columns = ["Event_id", "dPSI", "pval"]
    return df[(df["pval"] < 0.05) & (df["dPSI"].abs() >= 0.1)].copy()

def extract_gene_id(event_id):
    return event_id.split(";")[0]

def unique_to_variable(var_df, strict_df):
    strict_genes = set(strict_df["Event_id"].apply(extract_gene_id))
    var_df = var_df.copy()
    var_df["gene_id"] = var_df["Event_id"].apply(extract_gene_id)
    return var_df[~var_df["gene_id"].isin(strict_genes)].copy()

def style_table(df):
    return df.style\
        .set_properties(**{
            "font-size": "12px",
            "font-weight": "bold",
            "border": "1px solid #ddd",
            "padding": "6px 12px",
            "text-align": "center"
        })\
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#2196F3"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("padding", "6px 12px"),
                ("text-align", "center")
            ]},
            {"selector": "tr:nth-child(even)", "props": [
                ("background-color", "#f2f2f2")
            ]},
        ])\
        .hide(axis="index")

def export_table_png(df, filepath, title=""):
    fig, ax = plt.subplots(figsize=(len(df.columns) * 1.8, len(df) * 0.6 + 0.8))
    ax.axis("off")
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.6)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2196F3")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f2f2f2")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#dddddd")
    if title:
        ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved -> {filepath}")

print("Setup done!")

Setup done!


## Step 1 — Apply significance filters

We filter each diff file keeping only events where:
- p-value < 0.05 (statistically significant)
- |dPSI| ≥ 0.1 (biologically meaningful change, at least 10%)

In [54]:
# Apply to all events, runs and comparisons
results = {}
for comp_name, temp in COMPARISONS.items():
    results[comp_name] = {}
    for event in EVENTS:
        results[comp_name][event] = {
            "strict": load_and_filter(STRICT, event, temp),
            "var1":   load_and_filter(VAR1,   event, temp),
            "var5":   load_and_filter(VAR5,   event, temp),
        }

print("Filtering done!")

Filtering done!


## Step 2 — Summary table of significant event counts

Quick overview of how many events pass the filter per run and comparison.
This is the numerical reference before we look at which specific events differ.

In [55]:
rows = []
for comp_name in COMPARISONS:
    for event in EVENTS:
        rows.append({
            "Comparison": comp_name,
            "Event": event,
            "Strict": len(results[comp_name][event]["strict"]),
            "Variable 1nt": len(results[comp_name][event]["var1"]),
            "Variable 5nt": len(results[comp_name][event]["var5"]),
        })

df_summary = pd.DataFrame(rows)
df_summary["V1 - S"] = df_summary["Variable 1nt"] - df_summary["Strict"]
df_summary["V5 - S"] = df_summary["Variable 5nt"] - df_summary["Strict"]

display(style_table(df_summary))
export_table_png(df_summary,
                 "../../figures/plots/table_step2_summary.png",
                 "Significant events (p < 0.05, |dPSI| ≥ 0.1)")

Comparison,Event,Strict,Variable 1nt,Variable 5nt,V1 - S,V5 - S
CT8_vs_CT20,A3,36,73,75,37,39
CT8_vs_CT20,A5,35,63,60,28,25
CT8_vs_CT20,RI,11,62,61,51,50
CT20_vs_PerKO,A3,45,73,74,28,29
CT20_vs_PerKO,A5,47,76,72,29,25
CT20_vs_PerKO,RI,13,64,65,51,52


Saved -> ../../figures/plots/table_step2_summary.png


## Step 3 — Find events unique to variable mode

The numerical subtraction above tells us *how many* extra events variable
mode finds. Now we find *which specific events* those are, events present
in variable but completely absent in strict.

This is done using a set difference on the Event_id column.

Note: since strict and variable use different coordinate systems, we compare
at the gene level (ENSMUSG ID) rather than the full event ID.

In [56]:
unique_events = {}
rows_unique = []

for comp_name in COMPARISONS:
    unique_events[comp_name] = {}
    for event in EVENTS:
        u_var1 = unique_to_variable(
            results[comp_name][event]["var1"],
            results[comp_name][event]["strict"]
        )
        u_var5 = unique_to_variable(
            results[comp_name][event]["var5"],
            results[comp_name][event]["strict"]
        )
        unique_events[comp_name][event] = {"var1": u_var1, "var5": u_var5}

        # Get V1-S and V5-S from df_summary for reference
        row_ref = df_summary[
            (df_summary["Comparison"] == comp_name) &
            (df_summary["Event"] == event)
        ].iloc[0]

        rows_unique.append({
            "Comparison":       comp_name,
            "Event":            event,
            "Unique to Var1nt": len(u_var1),
            "Unique to Var5nt": len(u_var5),
            "V1 - S (count)":   row_ref["V1 - S"],
            "V5 - S (count)":   row_ref["V5 - S"],
        })

df_unique = pd.DataFrame(rows_unique)
display(style_table(df_unique))
export_table_png(df_unique,
                 "../../figures/plots/table_step3_unique.png",
                 "Unique genes to variable mode vs strict")

Comparison,Event,Unique to Var1nt,Unique to Var5nt,V1 - S (count),V5 - S (count)
CT8_vs_CT20,A3,44,46,37,39
CT8_vs_CT20,A5,36,33,28,25
CT8_vs_CT20,RI,56,55,51,50
CT20_vs_PerKO,A3,45,46,28,29
CT20_vs_PerKO,A5,45,42,29,25
CT20_vs_PerKO,RI,55,56,51,52


Saved -> ../../figures/plots/table_step3_unique.png


## Step 4 — Output unique events to IOE files

We now look up these unique Event_ids in the original IOE files to get
their full IOE entries, and save them to new IOE files for downstream
analysis in IGV.

In [58]:
IOE_SUFFIXES = {
    "var1": ("variable_1", IOE_VAR1),
    "var5": ("variable_5", IOE_VAR5),
}

OUTPUT_DIR = "/Users/gricey/Desktop/Internship/data/ioe_diff"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for comp_name in COMPARISONS:
    for event in EVENTS:
        for var_key, (suffix, ioe_dir) in IOE_SUFFIXES.items():
            ioe_path = f"{ioe_dir}/events_{event}_{suffix}.ioe"
            ioe_df = pd.read_csv(ioe_path, sep="\t")

            unique_df = unique_events[comp_name][event][var_key]
            unique_gene_ids = set(unique_df["gene_id"])

            # Save IOE file
            ioe_unique = ioe_df[ioe_df["event_id"].apply(
                lambda x: x.split(";")[0]).isin(unique_gene_ids)]
            ioe_out = f"{OUTPUT_DIR}/{comp_name}_{event}_{var_key}_unique.ioe"
            ioe_unique.to_csv(ioe_out, sep="\t", index=False)

            # Save gene list as plain text
            gene_out = f"{OUTPUT_DIR}/{comp_name}_{event}_{var_key}_unique_genes.txt"
            with open(gene_out, "w") as f:
                for gene in sorted(unique_gene_ids):
                    f.write(gene + "\n")

            print(f"Saved {len(ioe_unique)} IOE entries and {len(unique_gene_ids)} genes -> {comp_name}_{event}_{var_key}")

Saved 76 IOE entries and 31 genes -> CT8_vs_CT20_A3_var1
Saved 72 IOE entries and 32 genes -> CT8_vs_CT20_A3_var5
Saved 51 IOE entries and 30 genes -> CT8_vs_CT20_A5_var1
Saved 47 IOE entries and 27 genes -> CT8_vs_CT20_A5_var5
Saved 100 IOE entries and 34 genes -> CT8_vs_CT20_RI_var1
Saved 98 IOE entries and 35 genes -> CT8_vs_CT20_RI_var5
Saved 72 IOE entries and 38 genes -> CT20_vs_PerKO_A3_var1
Saved 72 IOE entries and 38 genes -> CT20_vs_PerKO_A3_var5
Saved 66 IOE entries and 36 genes -> CT20_vs_PerKO_A5_var1
Saved 61 IOE entries and 33 genes -> CT20_vs_PerKO_A5_var5
Saved 112 IOE entries and 42 genes -> CT20_vs_PerKO_RI_var1
Saved 112 IOE entries and 42 genes -> CT20_vs_PerKO_RI_var5
